## Lecture fichier txt des arks

In [ ]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import time
import ast
import json
import urllib.request, urllib.error, urllib.parse
import os

In [ ]:
headers= {"User-Agent": "Mozilla/5.0"}
cookie = {"JSESSIONID": ""}

In [ ]:
start_date = 1883
end_date = 1889

Faire de ma liste de revues liste des numéros (via liste des revues par année)

In [ ]:
chemin1 = './arks_revues.json'
with open(chemin1, mode='r', encoding='utf-8') as f : 
    dico_revues = ast.literal_eval(f.read())

In [ ]:
dico_arks = { "numeros" : [],"urls" : []}
for revue in dico_revues : 
    url_revue = dico_revues[revue]
    for i in tqdm(range(start_date, end_date)):
        for j in tqdm(range(1, 13)): #éventuellement raffiner avec boucle sur les jours du mois
            url = url_revue+f"/date{i}{j:02d}01"
            response = requests.get(url, headers=headers, cookies=cookie)
            print(response.url)
            if response.status_code == 200 and response.url[-5:] == ".item" :
                dico_arks["urls"].append(response.url[23:-5])
                dico_arks["numeros"].append(revue+url[-8:])

In [ ]:
with open("arks_numeros.json", "w", encoding="utf-8") as out: 
    json.dump(dico_arks, out)

## scraping gallica des pdfs (ou jpeg??!!) avec enregistrement dans répertoire 

In [ ]:
chemin2 = './arks_numeros.json'#liste de listes ?
with open(chemin2, mode='r', encoding='utf-8') as f : 
    dico_arks2 = ast.literal_eval(f.read())

### en faire un tableau intermédiaire

In [ ]:
df_arks = pd.DataFrame(dico_arks2)
open('tableau_arks.csv', mode='w', encoding='utf-8').write(df_arks.to_csv())

In [ ]:
dico_arks2

## Récupérer jpg gallica (non)

In [ ]:
for numero, ark in tqdm(zip(dico_arks2['numeros'], dico_arks2['urls'])) :
    os.mkdir("/Users/mathieu/Documents/memoire/test_ocr/"+numero)
    dir_local = "/Users/mathieu/Documents/memoire/test_ocr/"+numero
    for page in tqdm(range(1, 1001)) :
        time.sleep(2)
        jpgfile = os.path.join(dir_local, numero + "_" + str(page) + ".jpg")
        url_jpg = 'http://gallica.bnf.fr/iiif/' + ark + '/f' + str(page) + '/full/3000/0/native.jpg'
        response = requests.get(url_jpg, headers=headers)
        if response.status_code == 200 : 
            time.sleep(2)
            urllib.request.urlretrieve(url_jpg, jpgfile)      
    time.sleep(10)
      



## Récupérer pdf gallica

In [ ]:

for numero, ark in tqdm(zip(dico_arks2['numeros'], dico_arks2['urls'])) :
    os.mkdir("/Users/mathieu/Documents/memoire/test_ocr/"+numero)
    dir_local = "/Users/mathieu/Documents/memoire/test_ocr/"+numero
    pdf_file = os.path.join(dir_local, numero + ".pdf")
    url_pdf = 'http://gallica.bnf.fr/' + ark + '.pdf'
    response = requests.get(url_pdf, headers=headers, cookies=cookie)
    time.sleep(1)
    if response.status_code == 200 : 
        urllib.request.urlretrieve(url_pdf, pdf_file)  
        time.sleep(1)
        print('un nv pdf téléchargé ! ')
          



# En // faire un csv via panda avec métadonnées et liens arks_numeros ?!

## from pdf to jpegs

In [ ]:
import re
from pdf2image import convert_from_path, pdfinfo_from_path
from tqdm import tqdm

In [ ]:
path_pdf = "/Users/mathieu/Documents/memoire/test_ocr/"

In [ ]:
from glob import glob
files_name = []
for file in glob(path_pdf+"*"):
    print(file)
    files_name.append(file)

In [ ]:
files_name = ["/Users/mathieu/Documents/memoire/test_ocr/revue_scientifique18840101"]

In [ ]:
MAX_W, MAX_H = 1000, 1500
DPI1 = 150
POPPLER_PATH = "/opt/homebrew/bin"

def target_size(pdf_path, dpi, max_w, max_h):
    info = pdfinfo_from_path(pdf_path, poppler_path=POPPLER_PATH)
    # "Page size: 595.28 x 841.89 pts (A4)"
    m = re.search(r"([0-9.]+) x ([0-9.]+)", info.get("Page size", ""))
    if not m:
        return (max_w, max_h)  # fallback
    w_pts, h_pts = map(float, m.groups())
    w_px = w_pts / 72 * dpi
    h_px = h_pts / 72 * dpi
    scale = min(max_w / w_px, max_h / h_px, 1.0)
    return (int(w_px * scale), int(h_px * scale))



In [ ]:
import glob
for file in tqdm(files_name) :
    path = file+"/*.pdf"
    pdf = glob.glob(path) 
    print(pdf)
    size1 = target_size(pdf[0], DPI1, MAX_W, MAX_H)
    print(size1)

    convert_from_path(pdf[0],
        dpi=DPI1,
        output_folder=file, 
        poppler_path=POPPLER_PATH,
        fmt="jpeg",
        output_file="page", size=size1)
    
    